In [8]:
import sqlite3
import os

print("=== Scanning for SQLite Databases in Workspace ===")
for root, dirs, files in os.walk(r'd:\My Applications\Midnigth stories'):
    # Skip node_modules and .git
    if 'node_modules' in root or '.git' in root:
        continue
    for f in files:
        if f.endswith('.sqlite') or f.endswith('.db'):
            db_path = os.path.join(root, f)
            print(f"\nFound Database: {db_path} (Size: {os.path.getsize(db_path)} bytes)")
            try:
                conn = sqlite3.connect(db_path)
                cursor = conn.cursor()
                cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
                tables = [t[0] for t in cursor.fetchall() if not t[0].startswith('sqlite_')]
                print("Tables and Row Counts:")
                for table in tables:
                    try:
                        cursor.execute(f"SELECT COUNT(*) FROM {table}")
                        print(f"  - {table}: {cursor.fetchone()[0]} rows")
                    except Exception as e:
                        print(f"  - {table}: Error: {e}")
                conn.close()
            except Exception as e:
                print(f"Error reading {f}: {e}")


In [5]:
import sqlite3
import os

DB_PATH = 'data/stories.db'
OUTPUT = 'scratch/remote_books.sql'

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row
cursor = conn.cursor()

# Get all books
cursor.execute("""
    SELECT b.id, b.title, b.author, b.description, b.file_url, b.cover_image_url,
           b.file_type, b.status, b.visibility, b.channel_type, b.is_user_submission, b.submission_status,
           c.slug as cat_slug
    FROM books b
    LEFT JOIN book_categories bc ON bc.book_id = b.id
    LEFT JOIN categories c ON c.id = bc.category_id
    ORDER BY b.id
""")
rows = cursor.fetchall()
conn.close()

lines = []
lines.append('-- SQL Script to insert uploaded books into remote Cloudflare D1')
lines.append('PRAGMA foreign_keys = OFF;')
lines.append('')

seen_books = set()
for row in rows:
    bid = row['id']
    
    # Sanitize text fields for SQL
    def esc(v):
        if v is None: return 'NULL'
        return "'" + str(v).replace("'", "''") + "'"
    
    if bid not in seen_books:
        seen_books.add(bid)
        lines.append(
            f"INSERT OR IGNORE INTO books (id, title, author, description, file_url, cover_image_url, file_type, status, visibility, channel_type, is_user_submission, submission_status) "
            f"VALUES ({bid}, {esc(row['title'])}, {esc(row['author'])}, {esc(row['description'])}, "
            f"{esc(row['file_url'])}, {esc(row['cover_image_url'])}, {esc(row['file_type'])}, "
            f"'published', 'public', {esc(row['channel_type'])}, 0, 'approved');"
        )
    
    # Use slug-based subquery so category_id matches the remote DB's auto-generated IDs
    cat_slug = row['cat_slug']
    if cat_slug:
        lines.append(
            f"INSERT OR IGNORE INTO book_categories (book_id, category_id) "
            f"SELECT {bid}, id FROM categories WHERE slug = {esc(cat_slug)} LIMIT 1;"
        )

lines.append('')
lines.append('PRAGMA foreign_keys = ON;')

with open(OUTPUT, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))

print(f'✅ Generated {OUTPUT} with {len(seen_books)} books using slug-based category lookups.')
print(f'   Total lines: {len(lines)}')

✅ Generated scratch/remote_books.sql with 430 books using slug-based category lookups.
   Total lines: 865


In [ ]:
# [Notebook MCP] Internal check: Waking up kernel connection or awaiting UI Kernel Selection...